# 02 — Image Models (ResNet50 · VGG16 · Custom CNN)
Train all three models on Intel Image dataset and compare results.

## 1. Mount Drive and load dataset

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Load dataset from Drive into fast local SSD
import os
if not os.path.exists('/content/intel_data'):
    print('Copying dataset from Drive...')
    os.system('cp -r /content/drive/MyDrive/ContentRecognition/image_dataset /content/intel_data')
    print('Done!')
else:
    print('Dataset already on local SSD.')

## 2. Imports

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import time
from sklearn.metrics import classification_report, confusion_matrix

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

## 3. Config

In [ ]:
CLASS_NAMES  = ['buildings', 'forest', 'glacier', 'mountain', 'sea', 'street']
NUM_CLASSES  = len(CLASS_NAMES)
BATCH_SIZE   = 32
EPOCHS       = 10
BASE_DIR     = '/content/drive/MyDrive/ContentRecognition'
CKPT_DIR     = f'{BASE_DIR}/checkpoints/image'
RESULTS_DIR  = f'{BASE_DIR}/results/image'

import os
os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

## 4. Data transforms and loaders

In [ ]:
train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.3, contrast=0.3),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

val_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

# NOTE: dataset path uses the Intel folder structure:
# seg_train/seg_train/<class>/<images>
# seg_test/seg_test/<class>/<images>
train_dataset = datasets.ImageFolder(
    '/content/intel_data/seg_train/seg_train',
    transform=train_transforms
)
val_dataset = datasets.ImageFolder(
    '/content/intel_data/seg_test/seg_test',
    transform=val_transforms
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE,
                          shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=2, pin_memory=True)

print(f'Train samples : {len(train_dataset)}')
print(f'Val   samples : {len(val_dataset)}')
print(f'Classes       : {CLASS_NAMES}')

## 5. Model definitions
Three separate factory functions — clean, no duplication.

In [ ]:
# -----------------------------------------------------------
# Custom CNN  (trained from scratch)
# -----------------------------------------------------------
class CustomCNN(nn.Module):
    """4-block CNN built from scratch. Each block: Conv-BN-ReLU x2 → MaxPool → Dropout."""

    def _block(self, in_ch, out_ch):
        return nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1), nn.BatchNorm2d(out_ch), nn.ReLU(),
            nn.Conv2d(out_ch, out_ch, 3, padding=1), nn.BatchNorm2d(out_ch), nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Dropout2d(0.25)
        )

    def __init__(self, num_classes=6):
        super().__init__()
        self.block1 = self._block(3,   32)   # 224 → 112
        self.block2 = self._block(32,  64)   # 112 →  56
        self.block3 = self._block(64,  128)  #  56 →  28
        self.block4 = self._block(128, 256)  #  28 →  14
        # 14 × 14 × 256 = 50176
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(50176, 512), nn.ReLU(), nn.Dropout(0.5),
            nn.Linear(512, 128),   nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.block4(x)
        return self.classifier(x)


# -----------------------------------------------------------
# ResNet50  (pretrained backbone, new head)
# -----------------------------------------------------------
def build_resnet50(num_classes=6):
    model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
    for p in model.parameters():
        p.requires_grad = False          # freeze backbone
    model.fc = nn.Sequential(
        nn.Linear(model.fc.in_features, 256),
        nn.ReLU(), nn.Dropout(0.4),
        nn.Linear(256, num_classes)
    )
    return model


# -----------------------------------------------------------
# VGG16  (pretrained backbone, new classifier head)
# -----------------------------------------------------------
def build_vgg16(num_classes=6):
    model = models.vgg16(weights=models.VGG16_Weights.IMAGENET1K_V1)
    for p in model.parameters():
        p.requires_grad = False          # freeze backbone
    # VGG classifier[6] is the final Linear(4096 → 1000)
    model.classifier[6] = nn.Sequential(
        nn.Linear(4096, 256),
        nn.ReLU(), nn.Dropout(0.4),
        nn.Linear(256, num_classes)
    )
    return model


print('Model definitions ready.')

## 6. Shared training function

In [ ]:
def train_model(model, model_name, epochs=EPOCHS):
    """Train model, save best checkpoint, return history and best accuracy."""
    model = model.to(device)
    save_path = f'{CKPT_DIR}/{model_name}.pth'

    # Only parameters that need gradients are updated
    optimizer = optim.Adam(
        filter(lambda p: p.requires_grad, model.parameters()), lr=1e-3
    )
    # Reduce LR by 10× every 5 epochs
    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.1)
    criterion = nn.CrossEntropyLoss()

    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
    best_val_acc = 0.0

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'\n{"="*52}')
    print(f'  Training : {model_name}  |  Trainable params: {trainable:,}')
    print(f'{"="*52}')

    for epoch in range(epochs):
        t0 = time.time()
        for phase in ['train', 'val']:
            model.train() if phase == 'train' else model.eval()
            loader = train_loader if phase == 'train' else val_loader

            running_loss, running_corrects = 0.0, 0

            for inputs, labels in loader:
                inputs, labels = inputs.to(device), labels.to(device)
                optimizer.zero_grad()

                with torch.set_grad_enabled(phase == 'train'):
                    outputs = model(inputs)
                    loss    = criterion(outputs, labels)
                    _, preds = torch.max(outputs, 1)
                    if phase == 'train':
                        loss.backward()
                        optimizer.step()

                running_loss     += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels)

            if phase == 'train':
                scheduler.step()

            epoch_loss = running_loss / len(loader.dataset)
            epoch_acc  = running_corrects.double() / len(loader.dataset)

            history[f'{phase}_loss'].append(epoch_loss)
            history[f'{phase}_acc'].append(epoch_acc.item())

            if phase == 'val' and epoch_acc > best_val_acc:
                best_val_acc = epoch_acc
                torch.save(model.state_dict(), save_path)

        print(
            f'  Epoch {epoch+1:02d}/{epochs} | '
            f'Train {history["train_acc"][-1]:.4f} | '
            f'Val {history["val_acc"][-1]:.4f} | '
            f'{time.time()-t0:.1f}s'
        )

    print(f'  ✅ Best Val Acc: {best_val_acc:.4f}  →  saved to {save_path}')
    return model, history, best_val_acc.item(), save_path


print('train_model() defined.')

## 7. Train all three models

In [ ]:
# Custom CNN
custom_cnn = CustomCNN(num_classes=NUM_CLASSES)
custom_cnn, hist_custom, acc_custom, path_custom = train_model(custom_cnn, 'CustomCNN')

# ResNet50
resnet = build_resnet50(num_classes=NUM_CLASSES)
resnet, hist_resnet, acc_resnet, path_resnet = train_model(resnet, 'ResNet50')

# VGG16
vgg = build_vgg16(num_classes=NUM_CLASSES)
vgg, hist_vgg, acc_vgg, path_vgg = train_model(vgg, 'VGG16')

results = {'Custom CNN': acc_custom, 'ResNet50': acc_resnet, 'VGG16': acc_vgg}

## 8. Evaluate all three models

In [ ]:
def evaluate(model, save_path, model_name):
    """Load best weights, run on val set, print classification report."""
    model.load_state_dict(torch.load(save_path, map_location=device))
    model.eval().to(device)

    all_preds, all_labels = [], []
    with torch.no_grad():
        for inputs, labels in val_loader:
            outputs = model(inputs.to(device))
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.numpy())

    print(f'\n{"="*52}')
    print(f'  {model_name} — Classification Report')
    print(f'{"="*52}')
    print(classification_report(all_labels, all_preds, target_names=CLASS_NAMES))
    return all_preds, all_labels


preds_custom, labels_ = evaluate(custom_cnn, path_custom, 'Custom CNN')
preds_resnet, _       = evaluate(resnet,      path_resnet, 'ResNet50')
preds_vgg,    _       = evaluate(vgg,          path_vgg,   'VGG16')

## 9. Plots — accuracy comparison, training curves, confusion matrices

In [ ]:
COLORS = {'Custom CNN': '#E74C3C', 'ResNet50': '#2B5BA8', 'VGG16': '#27AE60'}

# ── Plot A: Accuracy bar + Validation curves ──────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

names = list(results.keys())
accs  = [v * 100 for v in results.values()]
bars  = axes[0].bar(names, accs, color=list(COLORS.values()), width=0.5)
for bar, acc in zip(bars, accs):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                 f'{acc:.1f}%', ha='center', va='bottom', fontweight='bold')
axes[0].set_title('Validation Accuracy Comparison', fontweight='bold')
axes[0].set_ylabel('Accuracy (%)')
axes[0].set_ylim([0, 108])
axes[0].grid(axis='y', alpha=0.3)

for hist, name in zip([hist_custom, hist_resnet, hist_vgg], names):
    axes[1].plot(hist['val_acc'], label=name, color=COLORS[name], marker='o')
axes[1].set_title('Validation Accuracy per Epoch', fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/model_comparison.png', dpi=150)
plt.show()

# ── Plot B: Individual train/val curves for each model ────────────
fig, axes = plt.subplots(3, 2, figsize=(14, 14))
for row, (hist, name) in enumerate(zip(
    [hist_custom, hist_resnet, hist_vgg], names
)):
    c = COLORS[name]
    axes[row][0].plot(hist['train_acc'], label='Train', color=c, marker='o')
    axes[row][0].plot(hist['val_acc'],   label='Val',   color=c, linestyle='--', marker='s')
    axes[row][0].set_title(f'{name} — Accuracy', fontweight='bold')
    axes[row][0].legend(); axes[row][0].grid(True, alpha=0.3)

    axes[row][1].plot(hist['train_loss'], label='Train', color=c, marker='o')
    axes[row][1].plot(hist['val_loss'],   label='Val',   color=c, linestyle='--', marker='s')
    axes[row][1].set_title(f'{name} — Loss', fontweight='bold')
    axes[row][1].legend(); axes[row][1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/training_curves.png', dpi=150)
plt.show()

# ── Plot C: Confusion matrices ────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(21, 6))
for ax, preds, name in zip(
    axes,
    [preds_custom, preds_resnet, preds_vgg],
    names
):
    cm = confusion_matrix(labels_, preds)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax)
    ax.set_title(f'{name}', fontweight='bold')
    ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
    plt.setp(ax.get_xticklabels(), rotation=45)

plt.suptitle('Confusion Matrices', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/confusion_matrices.png', dpi=150)
plt.show()

print('\nAll plots saved!')

## 10. Summary table

In [ ]:
print(f'\n{"="*45}')
print(f'{"MODEL":<15} {"VAL ACCURACY":>15} {"CHECKPOINT":>12}')
print(f'{"="*45}')
for name, acc in results.items():
    print(f'{name:<15} {acc*100:>14.2f}%')
print(f'{"="*45}')
print('\nBest model checkpoint paths:')
for name, path in zip(names, [path_custom, path_resnet, path_vgg]):
    print(f'  {name}: {path}')